# PS LiDAR - Development Playground

**Available Bricks:**
- Brick 1: Data Loading
- Brick 2: Circular Clipping (manual coordinates)
- Brick 3: Normalisation Analysis
- Brick 4: Ground Filtering
- Brick 5: Height Normalisation + Export Checkpoint
- Brick 6: 3D Visualisation
- Brick 7: **Trunk Extraction** (verticality + DBSCAN + PCA)
- Brick 8: **Branch Extraction** (linearity + connectivity)
- Brick 9: **Export & Visualisation**


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import numpy as np
from pathlib import Path

# Resolve project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core import (
    PointCloudLoader,
    clip_circular_plot,
    detect_normalization,
    classify_ground,
    normalize_heights,
    export_point_cloud,
)

print(f"Project root: {project_root}")
print("All imports loaded successfully.")


Project root: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR
All imports loaded successfully.


In [2]:
FILE_PATH = "D:/LiDAR QAs/T460298_subsampled_laz1_4.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"File: {meta['filename']}")
print(f"Points: {meta['point_count']:,}")
print(f"Size: {meta['file_size_mb']} MB")

File: T460298_subsampled_laz1_4.laz
Points: 38,249,740
Size: 449.65 MB


In [3]:
# Load XYZ and available scalar fields
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f" {field}: {len(scalar_fields[field]):,} values")
    except:
        print(f" {field}: not available")

print(f"\nXYZ memory: {xyz_full.nbytes / (1024**2):.1f} MB")

 intensity: 38,249,740 values
 return_number: 38,249,740 values
 number_of_returns: 38,249,740 values
 classification: 38,249,740 values

XYZ memory: 437.7 MB


---
## 2. Circular Clipping (Brick 2)

**Instructions:**
1. Open the original file in CloudCompare
2. Use the "Point Picking" tool to locate the mat centre
3. Copy the Xg, Yg coordinates shown
4. Paste the values into `CENTER_X` and `CENTER_Y` below

In [1]:
# 
# USER PARAMETERS - Modify per plot
# 

# Centre coordinates (from CloudCompare Point Picking)
CENTER_X = -0.311372
CENTER_Y = -1.461118

# Plot radius in metres
PLOT_RADIUS = 16.0

# 

print(f"Centre: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radius: {PLOT_RADIUS}m")

Centre: (-0.311372, -1.461118)
Radius: 16.0m


In [5]:
# Run circular clipping
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f" Clipped in {elapsed*1000:.0f}ms")
print(f"Original points: {len(xyz_full):,}")
print(f"Points in plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

 Clipped in 712ms
Original points: 38,249,740
Points in plot: 21,502,878 (56.2%)


In [6]:
# Apply clipping to XYZ and scalar fields
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Scalar fields: {list(plot_scalars.keys())}")

# Free memory
del xyz_full, scalar_fields
import gc; gc.collect()
print(" Memory freed")

Plot XYZ: (21502878, 3)
Scalar fields: ['intensity', 'return_number', 'number_of_returns', 'classification']
 Memory freed


---
## 3. Normalisation Analysis (Brick 3)

In [7]:
analysis = detect_normalization(xyz)
print(f"Status: {analysis.status.value.upper()}")
print(f"Normalised: {analysis.is_normalized}")
print(f"Z range: {analysis.z_min:.2f}m to {analysis.z_max:.2f}m")

Estatus: NOT_NORMALIZED
Normalised?: False
Rango Z: -3.91m a 32.66m


---
## 4. Ground Filtering (Brick 4)

In [10]:
from src.core.ground import classify_ground
print("Running CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f" Completed in {time.perf_counter() - t0:.2f}s")
print(f"Ground: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetation: {ground_result.n_off_ground:,}")

# Extract ground and vegetation point clouds
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

print(f"Ground points: {len(ground_xyz):,}")
print(f"Vegetation points: {len(vegetation_xyz):,}")


Running CSF...
 Completed in 11.53s
Ground: 3,988,415 (18.5%)
Vegetation: 17,514,463
Ground points: 3,988,415
Vegetation points: 17,514,463


In [11]:
# Separate ground and vegetation
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Ground: {len(ground_xyz):,} points")
print(f"Vegetation: {len(vegetation_xyz):,} points")

Ground: 3,988,415 points
Vegetation: 17,514,463 points


---
## 5. Height Normalisation (Brick 5)

In [12]:
print("Normalising heights...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f" Completed in {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetation: Z = {veg_normalized[:, 2].min():.2f}m to {veg_normalized[:, 2].max():.2f}m")
print(f"Ground: Z = {ground_normalized[:, 2].min():.2f}m to {ground_normalized[:, 2].max():.2f}m")

Normalising heights...
 Completed in 2899ms

Vegetation: Z = -1.23m a 34.44m
Ground: Z = -0.90m a 0.82m


---
## 5.5 Export Checkpoint

In [15]:
from pathlib import Path
# Output directory
OUTPUT_DIR = Path("D:/OUTPUTS/T460298A")

# Export vegetation
veg_file = Path("D:/OUTPUTS/T460298A_VEG_NORM.laz")
export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"Ã¢Å“â€œ Vegetation: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Export ground
ground_file = Path("D:/OUTPUTS/T460298A_GROUND_NORM.laz")
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f" Ground: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

NameError: name 'Path' is not defined

---
## 6. 3D Visualisation (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colour by height
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Cloud: {len(pcd.points):,} points")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Normalised Vegetation", width=1280, height=720)


---
---
# BRICK 7: Trunk Extraction

**Pipeline** (dendromatics/3DFin):
1. Extract horizontal stripe at breast height
2. Voxelise and compute verticality (pgeof C++)
3. DBSCAN clustering in 2D (XY)
4. Iterative peeling
5. PCA for tree axes
6. Assign all points to the nearest axis

**Field parameters** (measured before/after scanning):


In [ ]:
# ==============================================================
# BRICK 7: TRUNK EXTRACTION
# ==============================================================

import os, sys, time
import numpy as np
import laspy
from pathlib import Path

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core.trunk_extraction import extract_trunks, TrunkExtractionConfig

# ---- FIELD-MEASURED PARAMETERS (modify per plot) ----

# Tree dimensions
DBH_MIN = 0.15          # metres smallest expected stem diameter
DBH_MAX = 0.8          # metres largest expected stem diameter
HEIGHT_MAX = 36.0       # metres tallest tree in the plot

# Breast-height detection band
STRIPE_LOWER = 8.0      # metres lower limit of detection stripe
STRIPE_UPPER = 11.0      # metres upper limit of detection stripe

# Crown distances (horizontal distance from trunk to tip of longest
# branch, measured per cardinal direction on the central tree)
CROWN_DISTANCE_A = 8.0  # metres North
CROWN_DISTANCE_B = 7.0  # metres East
CROWN_DISTANCE_C = 6.5  # metres South
CROWN_DISTANCE_D = 7.5  # metres West

# Derived: max crown radius = axis assignment distance
MAX_AXIS_DISTANCE = max(CROWN_DISTANCE_A, CROWN_DISTANCE_B,
                        CROWN_DISTANCE_C, CROWN_DISTANCE_D)
# Derived: stem search radius (half the max DBH, with a small margin)
STEM_SEARCH_RADIUS = (DBH_MAX / 2) + 0.40

print(f"Crown distances: A={CROWN_DISTANCE_A}m, B={CROWN_DISTANCE_B}m, "
      f"C={CROWN_DISTANCE_C}m, D={CROWN_DISTANCE_D}m")
print(f"Max axis distance (derived): {MAX_AXIS_DISTANCE}m")
print(f"Stem search radius (derived): {STEM_SEARCH_RADIUS}m")

# ---- ALGORITHM PARAMETERS ----

VOXEL_RESOLUTION = 0.05   # metres
VERTICALITY_THRESH = 0.7  # 0-1 (higher = stricter)
PEELING_ITERATIONS = 3  # passes of verticality peeling
MIN_CLUSTER_PTS = 500     # minimum voxels per stem cluster

# ---- OPTIONAL BASAL ANCHOR REFINEMENT ----
AXIS_REFINEMENT_MODE = "basal_anchor"  # "none" | "basal_anchor"
BASAL_ANCHOR_MIN_HEIGHT = 0.15
BASAL_ANCHOR_MAX_HEIGHT = 0.80
BASAL_ANCHOR_GAP_TO_STRIPE = 0.05
BASAL_ANCHOR_SLICE_STEP = 0.05
BASAL_ANCHOR_SLICE_HALF_WIDTH = 0.03
BASAL_ANCHOR_CLUSTER_EPS = 0.03
BASAL_ANCHOR_MIN_POINTS = 60
BASAL_ANCHOR_MIN_SUPPORT_SLICES = 3
BASAL_ANCHOR_SEARCH_RADIUS_FACTOR = 1.5
BASAL_ANCHOR_SEARCH_RADIUS_MIN = 0.20
BASAL_ANCHOR_SEARCH_RADIUS_MAX = 0.60
BASAL_ANCHOR_RADIUS_MIN_FACTOR = 0.80
BASAL_ANCHOR_RADIUS_MAX_FACTOR = 1.35
BASAL_ANCHOR_MIN_ARC_COVERAGE = 0.20
BASAL_ANCHOR_MAX_FIT_RESIDUAL_RATIO = 0.12
BASAL_ANCHOR_MAX_CENTER_DRIFT = 0.08
BASAL_ANCHOR_MAX_RADIUS_CV = 0.25
BASAL_ANCHOR_MAX_AXIS_RATIO = 1.80
BASAL_ANCHOR_MATCH_MAX_XY_DISTANCE = 0.75
BASAL_ANCHOR_MATCH_MAX_TILT_DEG = 30.0
BASAL_ANCHOR_MATCH_RADIUS_RATIO_MIN = 0.60
BASAL_ANCHOR_MATCH_RADIUS_RATIO_MAX = 1.60

# ---- LOAD CHECKPOINT ----

VEG_CHECKPOINT = Path("J:/OUTPUTS/T460298A_VEG_NORM.laz")

print(f"\nLoading: {VEG_CHECKPOINT.name}")
t0 = time.perf_counter()
las = laspy.read(str(VEG_CHECKPOINT))
veg_normalized = np.column_stack([
    np.array(las.x, dtype=np.float32),
    np.array(las.y, dtype=np.float32),
    np.array(las.z, dtype=np.float32),
])
print(f"  {len(veg_normalized):,} points in {time.perf_counter()-t0:.1f}s")
print(f"  Z range: {veg_normalized[:,2].min():.2f}m to {veg_normalized[:,2].max():.2f}m")

# ---- RUN EXTRACTION ----

trunk_config = TrunkExtractionConfig(
    stripe_lower_limit=STRIPE_LOWER,
    stripe_upper_limit=STRIPE_UPPER,
    dbh_min=DBH_MIN,
    dbh_max=DBH_MAX,
    height_max=HEIGHT_MAX,
    max_axis_distance=MAX_AXIS_DISTANCE,
    stem_search_radius=STEM_SEARCH_RADIUS,
    # ---- CLUSTER VALIDATION (rejects understory/regeneration) ----
    cluster_circularity_min=0.15,       # min XY circularity
    cluster_diameter_max_factor=2.5,    # max cluster diam = DBH_MAX * factor
    cluster_min_height=2.0,             # min stripe height for small trees
    cluster_min_diameter=0.05,           # min diameter (metres)
    voxel_resolution_xy=VOXEL_RESOLUTION,
    voxel_resolution_z=VOXEL_RESOLUTION,
    verticality_threshold=VERTICALITY_THRESH,
    peeling_iterations=PEELING_ITERATIONS,
    min_cluster_points=MIN_CLUSTER_PTS,
    axis_refinement_mode=AXIS_REFINEMENT_MODE,
    basal_anchor_min_height=BASAL_ANCHOR_MIN_HEIGHT,
    basal_anchor_max_height=BASAL_ANCHOR_MAX_HEIGHT,
    basal_anchor_gap_to_stripe=BASAL_ANCHOR_GAP_TO_STRIPE,
    basal_anchor_slice_step=BASAL_ANCHOR_SLICE_STEP,
    basal_anchor_slice_half_width=BASAL_ANCHOR_SLICE_HALF_WIDTH,
    basal_anchor_cluster_eps=BASAL_ANCHOR_CLUSTER_EPS,
    basal_anchor_min_points=BASAL_ANCHOR_MIN_POINTS,
    basal_anchor_min_support_slices=BASAL_ANCHOR_MIN_SUPPORT_SLICES,
    basal_anchor_search_radius_factor=BASAL_ANCHOR_SEARCH_RADIUS_FACTOR,
    basal_anchor_search_radius_min=BASAL_ANCHOR_SEARCH_RADIUS_MIN,
    basal_anchor_search_radius_max=BASAL_ANCHOR_SEARCH_RADIUS_MAX,
    basal_anchor_radius_min_factor=BASAL_ANCHOR_RADIUS_MIN_FACTOR,
    basal_anchor_radius_max_factor=BASAL_ANCHOR_RADIUS_MAX_FACTOR,
    basal_anchor_min_arc_coverage=BASAL_ANCHOR_MIN_ARC_COVERAGE,
    basal_anchor_max_fit_residual_ratio=BASAL_ANCHOR_MAX_FIT_RESIDUAL_RATIO,
    basal_anchor_max_center_drift=BASAL_ANCHOR_MAX_CENTER_DRIFT,
    basal_anchor_max_radius_cv=BASAL_ANCHOR_MAX_RADIUS_CV,
    basal_anchor_max_axis_ratio=BASAL_ANCHOR_MAX_AXIS_RATIO,
    basal_anchor_match_max_xy_distance=BASAL_ANCHOR_MATCH_MAX_XY_DISTANCE,
    basal_anchor_match_max_tilt_deg=BASAL_ANCHOR_MATCH_MAX_TILT_DEG,
    basal_anchor_match_radius_ratio_min=BASAL_ANCHOR_MATCH_RADIUS_RATIO_MIN,
    basal_anchor_match_radius_ratio_max=BASAL_ANCHOR_MATCH_RADIUS_RATIO_MAX,
)

print(f"\nInput points: {len(veg_normalized):,}")
trunk_result = extract_trunks(veg_normalized, trunk_config, verbose=True)
basal_anchor_validated = int(trunk_result.tree_axes[0].get("basal_anchor_candidates_validated", 0)) if trunk_result.tree_axes else 0
basal_anchor_candidates = int(trunk_result.tree_axes[0].get("basal_anchor_candidates_total", 0)) if trunk_result.tree_axes else 0
basal_anchor_matches = sum(1 for ax in trunk_result.tree_axes if ax.get("basal_anchor_applied", False))
print(f"\nTrees found: {trunk_result.n_trees}")
print(f"basal-anchor validated: {basal_anchor_validated} of {basal_anchor_candidates} candidates")
print(f"basal-anchor matched trunks: {basal_anchor_matches} of {trunk_result.n_trees} trees")


Crown distances: A=8.0m, B=7.0m, C=6.5m, D=7.5m
Max axis distance (derived): 8.0m
Stem search radius (derived): 0.8m

Loading: T460298A_VEG_NORM.laz
  17,514,463 points in 16.0s
  Z range: -1.23m to 34.44m

Input points: 17,514,463
Trunk extraction: 17,514,463 points
  Stripe: 8.0m – 11.0m
  Verticality threshold: 0.7
  Peeling iterations: 3
  Min cluster points: 500
  Stripe points: 700,026
  Peeling iteration 1/3
    Voxels: 135,663
    After verticality filter: 365,704 pts (removed 334,322), 69,927 voxels
    Clusters kept: 31, points: 318,496
  Peeling iteration 2/3
    Voxels: 60,830
    After verticality filter: 295,218 pts (removed 23,278), 54,523 voxels
    Clusters kept: 37, points: 283,640
  Peeling iteration 3/3
    Voxels: 51,783
    After verticality filter: 279,753 pts (removed 3,887), 50,532 voxels
    Clusters kept: 35, points: 278,082
  Cluster validation (cross-section geometry):
    Cluster 0: ✗ REJECTED (not_circular: circ=0.056 < 0.15) [569 pts]
    Cluster 1: ✓ va

c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\src\core\trunk_extraction.py:457: RuntimeWarning: Number of calls to function has reached maxfev = 4000.
  params, _ = optimize.leastsq(_residuals, init, maxfev=4000)


In [2]:
# ==============================================================
# BRICK 7 (cont): SEPARATE TRUNK vs NON-TRUNK
# ==============================================================

# trunk_mask = points within stem_search_radius of an axis
# tree_ids = which tree each point belongs to (-1 = unassigned)

trunk_points = veg_normalized[trunk_result.trunk_mask]
non_trunk_mask = ~trunk_result.trunk_mask
non_trunk_points = veg_normalized[non_trunk_mask]

print(f"Trunk points:     {len(trunk_points):,} ({trunk_result.trunk_mask.mean():.1%})")
print(f"Non-trunk points: {len(non_trunk_points):,}")
print(f"Assigned to a tree: {(trunk_result.tree_ids >= 0).sum():,}")
print(f"Unassigned:         {(trunk_result.tree_ids == -1).sum():,}")


Trunk points:     17,514,078 (100.0%)
Non-trunk points: 385
Assigned to a tree: 17,514,463
Unassigned:         0


In [ ]:
# ==============================================================
# BRICK 7B: STEM CLEANING + SECTIONING (3DFin approach)
# ==============================================================
# Step 1: 2nd verticality pass - removes non-vertical material
#         (branches, attached understory, foliage)
# Step 2: Section-wise circle fitting - computes per-section
#         diameters for each tree

from src.core.trunk_validation import (
    clean_stems, compute_stem_sections, StemCleaningConfig,
    filter_trees, TreeFilterConfig
)

# ---- CONFIGURATION ----
STEM_CLEANING_MODE = "suspicious_only"  # "global" or "suspicious_only"

stem_config = StemCleaningConfig(
    mode=STEM_CLEANING_MODE,
    # 2nd verticality pass
    verticality_threshold=0.6,
    verticality_scale=0.1,
    voxel_resolution_xy=0.02,
    voxel_resolution_z=0.02,
    # Sectioning (like 3DFin)
    section_len=0.2,             # distance between sections (m)
    section_wid=0.05,            # half-width of section slice (m)
    min_points_section=80,       # min points for circle fitting
    r_min=DBH_MIN / 2,           # min valid radius
    r_max=DBH_MAX / 2,           # max valid radius
    n_sectors=16,
    min_sectors=9,
    sector_width=0.02,
    inner_circle_ratio=0.5,
    max_inner_points=5,
    minimum_height=0.3,          # lowest section (m)
    maximum_height=HEIGHT_MAX,   # highest section (m)
    cluster_eps=0.02,
)

# ---- STEP 1: STEM CLEANING ----
print(f"Input: {trunk_result.n_trees} trees, {trunk_result.trunk_mask.sum():,} trunk points")

cleaning_result = clean_stems(veg_normalized, trunk_result, stem_config)

print(f"  Cleaning mode used: {cleaning_result.mode_used}")
print(
    f"  Trees processed by 2nd pass: {cleaning_result.n_trees_processed}/"
    f"{trunk_result.n_trees}"
)
print(f"  Trees skipped: {cleaning_result.n_trees_skipped}")
print(
    f"  Points passed to 2nd verticality: "
    f"{cleaning_result.n_points_processed_verticality:,}"
)
print(f"  Global fallback activated: {cleaning_result.used_global_fallback}")

# ---- STEP 2: SECTIONING ----
section_result = compute_stem_sections(
    veg_normalized,
    cleaning_result.stem_mask,
    trunk_result.tree_ids,
    stem_config,
)

# ---- STEP 3: TREE-LEVEL FILTERS (Height & Edge) ----
filter_config = TreeFilterConfig(
    plot_center_x=CENTER_X,
    plot_center_y=CENTER_Y,
    max_distance_from_center=PLOT_RADIUS - 0.5,
)

filter_result = filter_trees(
    veg_normalized,
    cleaning_result.stem_mask,
    trunk_result.tree_ids,
    filter_config,
)

# ---- EXPORT cleaned stems and non-stem candidates for CloudCompare ----
from pathlib import Path

from src.core.features import compute_exportable_geometry_features
from src.core.io import export_point_cloud
from src.core.trunk_audit import build_trunk_audit_table, export_trunk_audit_table

OUTPUT_DIR = Path("J:/OUTPUTS/T460298A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trunk_audit_df = build_trunk_audit_table(
    veg_normalized,
    trunk_result,
    cleaning_result,
    section_result,
    center_x=CENTER_X,
    center_y=CENTER_Y,
)
audit_csv_path = export_trunk_audit_table(
    trunk_audit_df,
    OUTPUT_DIR / "trunk_audit27.csv",
)
print(f"✓ Trunk audit: {audit_csv_path.name} ({len(trunk_audit_df)} candidates)")

feature_scale = 0.10
feature_voxel_size = 0.05
feature_names = [
    "verticality",
    "linearity",
    "planarity",
    "sphericity",
    "anisotropy",
    "surface_variation",
    "roughness",
    "neighbor_count",
    "volume_density",
]

cache_dir = Path("cache/brick7_features")
cache_dir.mkdir(parents=True, exist_ok=True)
cache_key = f"{OUTPUT_DIR.name}_n{len(veg_normalized)}_s{feature_scale:.2f}_v{feature_voxel_size:.2f}"
feature_cache_path = cache_dir / f"{cache_key}.npz"

geometry_features = None
if feature_cache_path.exists():
    with np.load(feature_cache_path, allow_pickle=False) as cached:
        cache_valid = (
            int(cached["point_count"]) == len(veg_normalized)
            and np.isclose(float(cached["feature_scale"]), feature_scale)
            and np.isclose(float(cached["feature_voxel_size"]), feature_voxel_size)
            and all(name in cached.files for name in feature_names)
        )
        if cache_valid:
            geometry_features = {name: cached[name] for name in feature_names}
            print(f"\nLoaded Brick 7 geometry feature cache: {feature_cache_path}")

if geometry_features is None:
    print("\nComputing Brick 7 geometric features for export...")
    geometry_features = compute_exportable_geometry_features(
        veg_normalized,
        scale=feature_scale,
        voxel_resolution_xy=feature_voxel_size,
        voxel_resolution_z=feature_voxel_size,
        verbose=False,
    )
    cache_payload = {
        "point_count": np.array([len(veg_normalized)], dtype=np.int64),
        "feature_scale": np.array([feature_scale], dtype=np.float32),
        "feature_voxel_size": np.array([feature_voxel_size], dtype=np.float32),
    }
    for name in feature_names:
        cache_payload[name] = geometry_features[name].astype(np.float32, copy=False)
    np.savez_compressed(feature_cache_path, **cache_payload)
    print(f"Saved Brick 7 geometry feature cache: {feature_cache_path}")

final_stem_mask = filter_result.stem_mask
non_stem_mask = ~final_stem_mask

total_points = len(veg_normalized)
stem_points = veg_normalized[final_stem_mask]
stem_ids = filter_result.tree_ids[final_stem_mask]
stem_ids_clamped = np.clip(stem_ids, 0, 255).astype(np.uint8)
stem_features = {
    name: values[final_stem_mask]
    for name, values in geometry_features.items()
}

non_stem_points = veg_normalized[non_stem_mask]
non_stem_features = {
    name: values[non_stem_mask]
    for name, values in geometry_features.items()
}
non_stem_classification = np.zeros(len(non_stem_points), dtype=np.uint8)

stem_output = OUTPUT_DIR / "trunks_validated27.laz"
non_stem_output = OUTPUT_DIR / "non_stem_candidates27.laz"

export_point_cloud(
    stem_output,
    stem_points,
    classification=stem_ids_clamped,
    extra_dimensions=stem_features,
    point_format=6,
)
export_point_cloud(
    non_stem_output,
    non_stem_points,
    classification=non_stem_classification,
    extra_dimensions=non_stem_features,
    point_format=6,
)

stem_pct = len(stem_points) / max(total_points, 1)
non_stem_pct = len(non_stem_points) / max(total_points, 1)

print(f"\n✓ Exported: {stem_output}")
print(f"  Stem points: {len(stem_points):,} ({stem_pct:.1%})")
print(f"  Removed by cleaning: {cleaning_result.n_points_removed:,} non-vertical points")
print(f"\n✓ Exported: {non_stem_output}")
print(f"  Non-stem candidate points: {len(non_stem_points):,} ({non_stem_pct:.1%})")
print(f"  Partition total check: {len(stem_points) + len(non_stem_points):,} / {total_points:,}")


Input: 32 trees, 17,514,078 trunk points

Stem cleaning (2nd verticality pass):
  Mode request: suspicious_only
  Fallback to global: suspicious trees=32/32, suspicious points=17,514,078/17,514,078

Stem cleaning (2nd verticality pass):
  Mode: global
  Input: 17,514,078 trunk points
  Verticality threshold: 0.6
  Scale: 0.1m
  Computing verticality on 17,514,078 trunk points...
    Tree   0: 439,417 → 114,670 pts (-74%)
    Tree   1: 237,149 → 57,815 pts (-76%)
    Tree   2: 187,753 → 44,058 pts (-77%)
    Tree   3: 302,379 → 148,763 pts (-51%)
    Tree   4: 293,601 → 131,051 pts (-55%)
    Tree   5: 766,771 → 270,000 pts (-65%)
    Tree   6: 161,576 → 35,548 pts (-78%)
    Tree   7: 216,291 → 61,756 pts (-71%)
    Tree   8: 125,562 → 74,132 pts (-41%)
    Tree   9: 456,188 → 171,015 pts (-63%)
    Tree  10: 444,541 → 132,502 pts (-70%)
    Tree  11: 496,126 → 198,715 pts (-60%)
    Tree  12: 396,223 → 133,025 pts (-66%)
    Tree  13: 329,367 → 158,712 pts (-52%)
    Tree  14: 461,745

C:\Users\geoal\AppData\Local\Temp\ipykernel_52620\4211843771.py:126: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  int(cached["point_count"]) == len(veg_normalized)
C:\Users\geoal\AppData\Local\Temp\ipykernel_52620\4211843771.py:127: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  and np.isclose(float(cached["feature_scale"]), feature_scale)
C:\Users\geoal\AppData\Local\Temp\ipykernel_52620\4211843771.py:128: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  and np.isclo


Loaded Brick 7 geometry feature cache: cache\brick7_features\T460298A_n17514463_s0.10_v0.05.npz

✓ Exported: J:\OUTPUTS\T460298A\trunks_validated25.laz
  Stem points: 6,403,402 (36.6%)
  Removed by cleaning: 11,110,676 non-vertical points

✓ Exported: J:\OUTPUTS\T460298A\non_stem_candidates25.laz
  Non-stem candidate points: 11,111,061 (63.4%)
  Partition total check: 17,514,463 / 17,514,463


---
# BRICK 8: Branch Extraction

**Pipeline:**
1. Compute linearity (pgeof) on non-trunk points
2. Filter by linearity threshold
3. 26-neighbour connectivity graph
4. Keep only components connected to trunks
5. Filter by maximum branch length


In [ ]:
# ==============================================================
# BRICK 8: BRANCH EXTRACTION
# ==============================================================

from src.core.branch_extraction import extract_branches, BranchExtractionConfig

# ---- FIELD-MEASURED PARAMETERS ----
MAX_BRANCH_LENGTH = 8.0     # metres Ã¢â‚¬â€ longest expected branch

# ---- ALGORITHM PARAMETERS ----
LINEARITY_THRESH = 0.5      # 0-1 (higher = stricter)
CONNECTIVITY_RADIUS = 0.05  # metres (voxel size for connectivity graph)
MIN_BRANCH_POINTS = 50      # minimum points per branch cluster

branch_config = BranchExtractionConfig(
    max_branch_length=MAX_BRANCH_LENGTH,
    linearity_threshold=LINEARITY_THRESH,
    connectivity_radius=CONNECTIVITY_RADIUS,
    min_branch_points=MIN_BRANCH_POINTS,
)

print(f"Branch config: {branch_config}")
print(f"Input points: {len(veg_normalized):,}")
print(f"Trunk points: {trunk_result.trunk_mask.sum():,}")

branch_result = extract_branches(
    veg_normalized,
    trunk_result,
    branch_config,
    verbose=True
)

print(f"\nBranch points: {branch_result.n_branch_points:,}")
print(f"Wood (trunk+branch): {branch_result.wood_mask.sum():,} "
      f"({branch_result.wood_mask.mean():.1%})")


---
# BRICK 9: Export & Visualisation

Exports separated clouds for validation in CloudCompare.


In [ ]:
# ==============================================================
# BRICK 9: EXPORT & TREE INVENTORY
# ==============================================================

from src.core.io import export_point_cloud
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("D:/OUTPUTS/T460298A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Build tree_id classification for the entire wood cloud ---
# tree_ids: per-point tree assignment from Brick 7
# Clamp to uint8 range (0-254, 255 = unassigned)
tree_ids_clamped = trunk_result.tree_ids.copy()
tree_ids_clamped[tree_ids_clamped == -1] = 255
tree_ids_clamped = tree_ids_clamped.astype(np.uint8)

# --- Export LAZ files with tree IDs ---

# Trunks (with tree ID)
trunk_pts = veg_normalized[trunk_result.trunk_mask]
trunk_ids = tree_ids_clamped[trunk_result.trunk_mask]
export_point_cloud(OUTPUT_DIR / "trunks.laz", trunk_pts, classification=trunk_ids, point_format=6)
print(f"Ã¢Å“â€œ Trunks: {len(trunk_pts):,} points")

# Branches (with tree ID)
branch_pts = veg_normalized[branch_result.branch_only_mask]
branch_ids = tree_ids_clamped[branch_result.branch_only_mask]
export_point_cloud(OUTPUT_DIR / "branches.laz", branch_pts, classification=branch_ids, point_format=6)
print(f"Ã¢Å“â€œ Branches: {len(branch_pts):,} points")

# Wood structure (trunk + branches, with tree ID)
wood_pts = veg_normalized[branch_result.wood_mask]
wood_ids = tree_ids_clamped[branch_result.wood_mask]
export_point_cloud(OUTPUT_DIR / "wood_structure.laz", wood_pts, classification=wood_ids, point_format=6)
print(f"Ã¢Å“â€œ Wood structure: {len(wood_pts):,} points")

# --- Tree Inventory (Excel) ---

rows = []
for ax in trunk_result.tree_axes:
    tid = ax['tree_id']
    cx, cy = ax['centroid'][0], ax['centroid'][1]
    n_pts = (trunk_result.tree_ids == tid).sum()
    rows.append({
        'Tree_ID': tid,
        'X': round(float(cx), 3),
        'Y': round(float(cy), 3),
        'Z_min': round(float(ax['z_min']), 2),
        'Z_max': round(float(ax['z_max']), 2),
        'N_points': int(n_pts),
    })

df = pd.DataFrame(rows)
xlsx_path = OUTPUT_DIR / "tree_inventory.xlsx"
df.to_excel(xlsx_path, index=False, sheet_name='Tree Inventory')

print(f"\nÃ¢Å“â€œ Tree inventory: {xlsx_path.name} ({len(df)} trees)")
print(f"\nAll exports saved to {OUTPUT_DIR}")
df


### Visualisation (Open3D)


In [ ]:
# ==============================================================
# BRICK 9 (cont): 3D VISUALISATION
# ==============================================================

import open3d as o3d
import numpy as np

# Colour trunks brown, branches green
trunk_colours = np.tile([0.6, 0.4, 0.2], (len(veg_normalized[trunk_result.trunk_mask]), 1))
branch_colours = np.tile([0.2, 0.7, 0.3], (len(branch_result.branch_points), 1))

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(wood)
pcd.colors = o3d.utility.Vector3dVector(np.vstack([trunk_colours, branch_colours]))

o3d.visualization.draw_geometries([pcd], window_name="Wood Structure")


---
## Next Steps

- **Brick 10:** Per-tree analysis (DBH, height, sweep)
- **Brick 11:** Fork detection
- **Brick 12:** HQP classification of branches and spikes
